# Create a new Pump.fun metadata dataset

Run this notebook from top to bottom to collect a new dataset. Edit only the settings cell.

The collector watches Solana mainnet for successful Pump.fun token creations. The moment the collection starts is T0. Until the enrollment period ends, every new creation is saved and its metadata is requested. The same tokens are requested again when enrollment ends, and once more after the follow-up period. A failed metadata request stays in the dataset; it does not remove the token.

Chain transactions come from `solana-rpc.publicnode.com` over IPv4. Metadata comes from the URI written in the creation transaction. If that URI contains an IPFS identifier and the first request fails, the same identifier is requested from `pump.mypinata.cloud` and then `gateway.pinata.cloud`. Local and private addresses are refused. A successful response keeps eight fields: name, symbol, description, createdOn, image, website, twitter, and telegram.

**Open and run:** Open the repository page containing this notebook, choose **Code → Download ZIP**, then extract it. Install Python 3.12. On macOS/Linux, open a terminal in the extracted repository folder and run:

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install jupyterlab
.venv/bin/jupyter lab pilots/shilin_mvp/Pumpfun_v3_Collection_Tutorial.ipynb
```

JupyterLab opens this notebook in your browser. Select its Python kernel (the program that runs the code), then choose **Run → Run All Cells**. Keep the repository's `code` folder beside the notebook.


## Settings

`START_AT_UTC` is when the new collection begins. Leave it empty to start as soon as you run the next cell. To start later, enter a future UTC time ending in `Z`, for example in the format `YYYY-MM-DDTHH:MM:SSZ`. A past time is rejected, because this collector records tokens from the moment it starts and cannot rebuild an earlier day.

`ENROLLMENT_HOURS` is how many hours of new tokens to include. `FOLLOWUP_HOURS` is how many hours to keep observing those tokens after enrollment ends. The previous collection used 24 and 36. A shorter pair, such as 1 and 2, creates a smaller dataset and finishes sooner.

`OUTPUT_DIR` is the folder for this new dataset. Leave it empty and the notebook creates `pumpfun_data/live-<start time>` beside the `code` folder. If a collection stops before it finishes, paste that same folder here and run the cell again; the collector continues the unfinished dataset using its saved schedule.

`MAX_RUNTIME_MINUTES` limits this invocation, including setup and any scheduled wait. `0` uses the remaining schedule plus five minutes. For a short exercise, use `ENROLLMENT_HOURS=1/60`, `FOLLOWUP_HOURS=1/60`, and `MAX_RUNTIME_MINUTES=3`; inspect the completion status afterward.


Leave `RPC_ENDPOINT` empty to use the automatic connection order below. An RPC is a service that reads Solana transactions for this notebook. HTTP/3 is another way to connect to the same PublicNode service. The collector tries each connection up to twice, then tries the next row. It keeps a working connection and switches again if later transaction queries fail.

| Order | Service and connection | Wait after each successful transaction query |
|---|---|---|
| 1 | PublicNode using Shilin's original HTTPS client | 0.2 seconds |
| 2 | The same PublicNode URL using HTTP/3 | 0.2 seconds |
| 3 | Official Solana RPC using HTTPS | 3 seconds |

The official RPC's 3-second wait is 15 times the PublicNode wait. It reduces query frequency after the earlier rate-limit response, so scanning takes longer. Each request also takes time to travel over the network. If a service asks the collector to wait before retrying (`Retry-After`), the collector follows that wait. The console prints the selected service and interval; `API_SELECTION.json` and `rpc_connection_attempts.jsonl` save the connection choices.

You can enter a public HTTPS RPC URL to choose one service directly. Entering the PublicNode URL keeps the automatic three-row order. Resume with the same `RPC_ENDPOINT` setting; connection attempts start with the saved order and retain their earlier records.


In [ ]:
# Edit this cell, then run the collection cell below.
START_AT_UTC = ""             # empty = start now; or a future UTC time ending in Z
ENROLLMENT_HOURS = 24         # hours of new Pump.fun creations to enroll
FOLLOWUP_HOURS = 36           # hours of observation after enrollment ends
OUTPUT_DIR = ""               # empty = create pumpfun_data/live-<start>; paste a folder to resume
RPC_ENDPOINT = ""             # empty = automatic connection order; or a public HTTPS RPC
MAX_RUNTIME_MINUTES = 0       # 0 = remaining schedule plus five minutes


## Collect

The next cell prints the schedule, waits if you chose a future start, and then collects the dataset. Leave it running. It finishes after the last checkpoint has been attempted or the runtime limit is reached, then prints the dataset path, completion status, and a short count of what was saved. `INCOMPLETE` means collection still has unfinished work; inspect `run_status.json` and `run/cohort_status.json` in the printed folder.


In [ ]:
from __future__ import annotations

import csv
import io
import json
import subprocess
import sys
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

here = Path.cwd().resolve()
code_dirs = [
    folder
    for candidate in [here, *here.parents]
    for folder in (candidate / "code", candidate / "pilots" / "shilin_mvp" / "code")
    if (folder / "run_mvp.py").is_file()
]
if not code_dirs:
    raise FileNotFoundError("Cannot find code/run_mvp.py. Open this notebook from the complete Shilin repository.")
CODE = code_dirs[0]
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from mvp.tutorial import prepare_collection

# Install the live connection package in the notebook's Python environment.
from importlib.metadata import PackageNotFoundError, version
import subprocess
try:
    live_package_ready = version("curl_cffi") == "0.16.3"
except PackageNotFoundError:
    live_package_ready = False
if not live_package_ready:
    print("Installing the live connection package...")
    subprocess.run([sys.executable, "-m", "pip", "install", "curl_cffi==0.16.3"], check=True)




def read_table(path: Path) -> list[dict[str, str]]:
    raw = path.read_text(encoding="utf-8", errors="replace").replace("\x00", "")
    return list(csv.DictReader(io.StringIO(raw)))


collection = prepare_collection(
    CODE,
    mode="rpc",
    start_at_utc=START_AT_UTC,
    enrollment_hours=ENROLLMENT_HOURS,
    checkpoint_hours=[ENROLLMENT_HOURS, ENROLLMENT_HOURS + FOLLOWUP_HOURS],
    output_dir=OUTPUT_DIR,
    rpc_endpoint=RPC_ENDPOINT,
    max_runtime_minutes=MAX_RUNTIME_MINUTES,
)
output = collection.output
protocol = json.loads(collection.protocol_path.read_text(encoding="utf-8"))
clock_path = output / "run" / "cohort_clock.json"
if collection.resumed and clock_path.is_file():
    start_text = json.loads(clock_path.read_text(encoding="utf-8"))["collection_started_at"]
else:
    start_text = collection.start_at_utc or datetime.now(timezone.utc).isoformat()
start_at = datetime.fromisoformat(start_text.replace("Z", "+00:00"))
print("Resuming:", collection.resumed)
print("Dataset schedule")
for checkpoint, seconds in collection.offsets.items():
    scheduled_at = start_at + timedelta(seconds=seconds)
    print(" ", checkpoint, scheduled_at.strftime("%Y-%m-%dT%H:%M:%SZ"))
selected_rpc = protocol.get("requested_rpc_endpoint") or next(
    row["endpoint"] for row in protocol["api_candidates"] if row.get("access_decision") == "allowed"
)
print("First chain connection:", selected_rpc)
print("Automatic order: PublicNode HTTPS, PublicNode HTTP/3, official RPC")
print("Transaction-query pauses: PublicNode 0.2s; official RPC 3s")
print("Metadata: declared URI first, then registered same-CID gateways")
print("Maximum runtime (minutes):", collection.max_runtime_seconds / 60)
print("Dataset folder:", output, flush=True)

result = subprocess.run(collection.command, cwd=collection.code_dir)
if result.returncode != 0:
    raise RuntimeError("Collection stopped with an error. Set OUTPUT_DIR to the dataset folder printed above and run this cell again to continue it.")

release = output / "mvp_release"
validation = json.loads((release / "validation_results.json").read_text(encoding="utf-8"))
ledger = read_table(release / "coverage_ledger.csv")
print("New dataset:", output)
print("Validation:", validation.get("status"))
print("Events:", validation.get("event_count"))
print("Observations:", validation.get("plan_rows"))
print("Final states:", validation.get("final_state_counts"))
for checkpoint, count in sorted(Counter(row["checkpoint"] + " " + row["final_state"] for row in ledger).items()):
    print(" ", checkpoint, count)

connection = json.loads((output / "API_SELECTION.json").read_text())
print("Selected chain service:", connection.get("selected_endpoint"))
print("Selected connection:", connection.get("transport"))
print("Transaction-query pause (seconds):", connection.get("getTransaction_pause_seconds"))


## If the cell stops early

The dataset folder printed above contains the tokens and metadata attempts saved before the stop. Keep the entire folder and its sibling `<dataset-name>-protocol.json` and `<dataset-name>-source-register.csv` files. Paste the folder into `OUTPUT_DIR`, keep its original `RPC_ENDPOINT`, choose a sufficient runtime limit, and run the collection cell again. `Resuming: True` confirms recovery. The collector continues from the saved clock and schedule; requests record their actual execution times.

Use persistent storage and keep the computer awake for a long run. Full live-window coverage and the 24/60-hour run still require verification with the selected RPC.
